# Search for Examinations

In [ ]:
import sys
sys.path.append('..')
from credentials import *

from elasticsearch_utils import *

import duckdb
import pandas as pd
import numpy as np
import re

data_path = "../data/"
raw_data_path = data_path+'raw_data/'

## Load Patients of Interest

In [ ]:
cols = ['master_person_id', 'patient_identifier3', 'patient_identifier4', 'patient_identifier2']

inclusion_patients_df = pd.read_csv(os.path.join(data_path, "chronic_kidney_disease_refined_inclusion_patients.csv"))[cols]

## List of Examinations
- Systolic BP
- Diastolic BP
- BMI

## Retrieve blood pressure and BMI (LIMITs need to be removed before outputting this finally!) 

In [ ]:
with duckdb.connect() as conn:
    examinations_omop_df = conn.execute("""ATTACH 'omop.duckdb' as PROD; USE PROD;
                              SELECT DISTINCT m.master_person_id
                                     , m.master_visit_occurrence_id
                                     , m.measurement_date
                                     , m.measurement_datetime
                                     , CASE WHEN UPPER(m.measurement_source_value_name) LIKE '%BMI%' OR UPPER(m.measurement_source_value_name) LIKE '%BODY%MASS%' THEN 'BMI'
                                            WHEN UPPER(m.measurement_source_value_name) LIKE '%HEIGHT%' THEN 'Height'
                                            WHEN m.measurement_source_value_name IN ('Actual Weight (kg)', 'Dry Weight', 'Weight', 'Weight (Kg)', 'Weight (kg)', 'Weight (kg) at Follow up') THEN 'Weight'
                                            WHEN UPPER(m.measurement_source_value_name) IN ('BLOOD PRESSURE') THEN 'Blood Pressure'
                                            WHEN UPPER(m.measurement_source_value_name) IN ('BP DIASTOLIC', 'DIASTOLIC BP', 'R BP DIASTOLIC') THEN 'BP Diastolic'
                                            WHEN UPPER(m.measurement_source_value_name) IN ('R BP SYSTOLIC', 'SYSTOLIC BP', 'BP SYSTOLIC', 'SYSTOLIC BP (MMHG)', 'SYSTOLIC BLOOD PRESSURE (MMHG)') THEN 'BP Systolic'
                                            ELSE '' END AS measure
                                     , m.measurement_source_value_name AS measurement_name
                                     , m.value_source_value AS value
                                     , m.unit_source_value AS unit
                              FROM ext_measurement AS m
                                  INNER JOIN inclusion_patients_df AS ip
                                      ON m.master_person_id = ip.master_person_id
                              WHERE UPPER(m.measurement_source_value_name) LIKE '%BMI%' OR UPPER(m.measurement_source_value_name) LIKE '%BODY%MASS%' OR
                                    UPPER(m.measurement_source_value_name) LIKE '%HEIGHT%' OR
                                    UPPER(m.measurement_source_value_name) LIKE '%WEIGHT (KG)%' OR UPPER(m.measurement_source_value_name) = 'WEIGHT' OR UPPER(m.measurement_source_value_name) = 'DRY WEIGHT' OR
                                    UPPER(m.measurement_source_value_name) IN ('BLOOD PRESSURE', 'BP DIASTOLIC', 'BP SYSTOLIC', 'DIASTOLIC BP', 'DIASTOLIC BP >90 MMHG',
                                                                               'R BP DIASTOLIC', 'R BP SYSTOLIC', 'SYSTOLIC BLOOD PRESSURE (MMHG)', 'SYSTOLIC BP', 'SYSTOLIC BP (MMHG)')
                              ORDER BY m.master_person_id, m.measurement_datetime;""").df()

In [ ]:
examinations_omop_df.head()

## Export Data

In [ ]:
# --- Save Results ---
file_name = "20251204_examinations_search_results.csv"

examinations_omop_df.to_csv(f"{raw_data_path}/{file_name}", index=False)
print("✅ Results saved.")